In [2]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/kharar/eg_pt_property.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/kharar/egbs_demand_v1.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_34.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_79.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_38.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_6.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_60.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_44.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_83.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_107.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/kharar/ouput_demand_detail/output_89.csv
Loading: /home/prerna/Punjab/punjab-data-

In [4]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

46837
287351
6315889


In [5]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

287351


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,21745,PT-1503-011887,PT,1522540800000,1554076799000,ACTIVE,21745,PT_OWNER_EXEMPTION,0.0,0.0
1,21745,PT-1503-011887,PT,1522540800000,1554076799000,ACTIVE,21745,PT_TIME_REBATE,0.0,0.0
2,21745,PT-1503-011887,PT,1522540800000,1554076799000,ACTIVE,21745,PT_UNIT_USAGE_EXEMPTION,0.0,0.0
3,21745,PT-1503-011887,PT,1522540800000,1554076799000,ACTIVE,21745,PT_TAX,4500.0,4500.0
4,21745,PT-1503-011887,PT,1522540800000,1554076799000,ACTIVE,21745,PT_FIRE_CESS,0.0,0.0


In [6]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

         consumercode earliest_fy latest_fy
0      PT-1503-005758     2018-19   2024-25
1      PT-1503-005764     2018-19   2024-25
2      PT-1503-005947     2018-19   2024-25
3      PT-1503-005994     2018-19   2025-26
4      PT-1503-005998     2018-19   2025-26
...               ...         ...       ...
47091  PT-1503-999930     2019-20   2024-25
47092  PT-1503-999931     2020-21   2024-25
47093  PT-1503-999932     2020-21   2024-25
47094  PT-1503-999933     2020-21   2024-25
47095  PT-1503-999982     2020-21   2024-25

[47096 rows x 3 columns]


In [7]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount
0  PT-1503-005758     2018-19   2024-25              1011.20
1  PT-1503-005764     2018-19   2024-25               372.00
2  PT-1503-005947     2018-19   2024-25               743.94
3  PT-1503-005994     2018-19   2025-26               501.14
4  PT-1503-005998     2018-19   2025-26              1171.87


In [8]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted_current.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted_current.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


     consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0  PT-1503-005758     2018-19   2024-25              1011.20   
1  PT-1503-005764     2018-19   2024-25               372.00   
2  PT-1503-005947     2018-19   2024-25               743.94   
3  PT-1503-005994     2018-19   2025-26               501.14   
4  PT-1503-005998     2018-19   2025-26              1171.87   

   current_fy_taxamount  
0                  0.00  
1                  0.00  
2                  0.00  
3                501.14  
4               1171.87  


In [9]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                         id       propertyid   tenantid  \
0      3949d5ff-bf33-4d9c-ab4b-90f0b3393eb0  PT-1503-2032754  pb.kharar   
1      9dd602f3-761b-4928-bec1-ea97141af1ef  PT-1503-2025080  pb.kharar   
2      cb3beec6-1fb7-4d53-9194-292119386cc7  PT-1503-1544579  pb.kharar   
3      dfe6ca3c-06d5-463e-a3ff-5c504f4fd965  PT-1503-2110143  pb.kharar   
4      5893f98c-c156-4da2-bc19-5104aad93497  PT-1503-2110145  pb.kharar   
...                                     ...              ...        ...   
46832  616f468e-9f9c-43fd-b83e-3618cef07d2a   PT-1503-086007  pb.kharar   
46833  da124d4b-550c-4860-865c-1693f09acf2f  PT-1503-1994060  pb.kharar   
46834  fe051f92-aafe-4490-b70e-acec827ced30  PT-1503-1994078  pb.kharar   
46835  65eae68a-d39d-47d8-ae47-411633e83d08   PT-1503-635396  pb.kharar   
46836  04853ec9-a6c9-4ffa-a00e-1c723a53b8bd  PT-1503-1994085  pb.kharar   

       status          ownershipcategory              usagecategory  \
0      ACTIVE     INDIVIDUAL

In [10]:
property_result_merged.to_csv('Punjab_Data_Analysis_kharar_final_2.csv', index=False)